# Análise de Sentimento de Avaliações de E-commerce (B2W)

**Integrantes:** _(preencher com os nomes do grupo)_

**Fonte dos dados:** Dataset B2W de avaliações de e-commerce brasileiro
(https://www.kaggle.com/datasets/fredericods/ptbr-sentiment-analysis-datasets)

**Objetivo:** Classificar automaticamente avaliações de clientes de
e-commerce como positivas ou negativas, com base apenas no texto da
avaliação (`review_text`). Uma aplicação assim pode ser usada, por
exemplo, para priorizar o atendimento a clientes insatisfeitos ou
monitorar a reputação de produtos em tempo real.

**Atributo-alvo:** `polarity` — variável categórica binária
(0 = avaliação negativa, 1 = avaliação positiva).

**Atributos preditivos:** `review_text` (texto livre da avaliação).

**Tipo da tarefa:** Classificação binária. O atributo-alvo representa
duas categorias (positivo/negativo), e não um valor numérico contínuo,
portanto a tarefa não é de regressão.

In [ ]:
# Instalação das bibliotecas necessárias (executar uma única vez)
!pip install -q wordcloud unidecode kagglehub

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import string
import unidecode

from wordcloud import WordCloud

plt.style.use('default')
sns.set_palette("husl")

## Carregamento dos dados

> **Nota sobre a fonte dos dados:** o carregamento abaixo usa
> `kagglehub`, que baixa o dataset diretamente do Kaggle sem precisar
> de nenhum arquivo local. Caso o grupo prefira hospedar o CSV no
> próprio repositório do GitHub (mais simples de garantir que roda
> sem autenticação), basta substituir esta célula por:
>
> ```python
> dataset = pd.read_csv("https://raw.githubusercontent.com/<usuario>/<repo>/main/data/b2w.csv")
> ```

In [ ]:
import kagglehub

path = kagglehub.dataset_download('fredericods/ptbr-sentiment-analysis-datasets')
dataset = pd.read_csv(path + "/b2w.csv")

print(f"{dataset.shape[0]} linhas x {dataset.shape[1]} colunas")
dataset.head()

## 5.2 Compreensão dos dados

In [ ]:
dataset.info()

O dataset possui as colunas `review_text` (texto da avaliação),
`rating` (nota dada pelo cliente) e `polarity` (rótulo binário que
usaremos como alvo). A seguir verificamos valores ausentes,
duplicatas e a distribuição do atributo-alvo.

In [ ]:
null_texts = dataset.query("review_text.isna()")
null_polarity = dataset.query("polarity.isna()")

print(f"{len(null_texts)} textos nulos")
print(f"{len(null_polarity)} polaridades nulas")
print(f"{dataset.duplicated().sum()} linhas duplicadas")

In [ ]:
if len(dataset.query("polarity.isna()")) > 0:
    ratings_without_polarity = dataset.query("polarity.isna()")["rating"].unique()
    print(f"Ratings das avaliações sem polaridade: {sorted(ratings_without_polarity)}")
    for rating in sorted(ratings_without_polarity):
        count = len(dataset.query("polarity.isna() and rating == @rating"))
        print(f"Rating {rating}: {count:,} avaliações")

As polaridades ausentes concentram-se em avaliações com notas
intermediárias, onde a classificação entre positivo/negativo é
ambígua (um cliente pode dar nota 3 tanto para uma experiência
mediana-positiva quanto mediana-negativa). Por não haver um critério
claro para atribuir rótulo a esses casos, optamos por remover essas
linhas em vez de inferir a polaridade.

In [ ]:
all_polarities = dataset["polarity"].unique()

print("Distribuição do atributo-alvo:")
for pol in sorted([p for p in all_polarities if pd.notna(p)]):
    count = len(dataset.query("polarity == @pol"))
    percentage = (count / len(dataset)) * 100
    print(f"Polaridade {pol}: {count:,} avaliações ({percentage:.1f}%)")

nan_sum = dataset["polarity"].isna().sum()
if nan_sum > 0:
    percentage = (nan_sum / len(dataset)) * 100
    print(f"NaN: {nan_sum:,} avaliações ({percentage:.1f}%)")

O dataset original é **desbalanceado**, com clara predominância
de avaliações positivas sobre negativas — um padrão típico de reviews
de e-commerce, já que clientes satisfeitos tendem a avaliar com mais
frequência que clientes neutros ou insatisfeitos. Esse
desbalanceamento será tratado explicitamente na etapa de
pré-processamento (seção 5.4), via undersampling da classe
majoritária, para evitar que o modelo aprenda a simplesmente prever
sempre "positivo".

## 5.3 Análise exploratória

Nesta seção exploramos o conteúdo textual das avaliações antes de
qualquer tratamento, para entender que tipo de vocabulário domina
cada classe.

In [ ]:
def wordcloud(dataframe: pd.DataFrame, text_column: str, title: str = "Nuvem de Palavras"):
    every_word = ' '.join([text for text in dataframe[text_column]])

    word_cloud = WordCloud(
        width=800,
        height=500,
        max_font_size=80,
        background_color='black',
        colormap='viridis'
    ).generate(every_word)

    plt.figure(figsize=(12, 8))
    plt.imshow(word_cloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(title, fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()

In [ ]:
all_pos = dataset.query("polarity == 1.0")
wordcloud(all_pos, "review_text", "Nuvem de Palavras Positivas (texto bruto)")

In [ ]:
all_neg = dataset.query("polarity == 0.0")
wordcloud(all_neg, "review_text", "Nuvem de Palavras Negativas (texto bruto)")

Sem nenhum tratamento, as nuvens de palavras tendem a ser
dominadas por *stopwords* (artigos, preposições, pronomes) e não
revelam ainda diferenças claras entre as classes — isso motiva o
pré-processamento textual feito na seção 5.4.

In [ ]:
dataset["text_length"] = dataset["review_text"].str.len()
dataset.groupby("polarity")["text_length"].describe()

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=dataset, x="text_length", hue="polarity", bins=50, kde=True)
plt.title("Distribuição do tamanho dos textos por polaridade", fontsize=14, fontweight='bold')
plt.xlabel("Número de caracteres")
plt.ylabel("Frequência")
plt.tight_layout()
plt.show()

_(Após rodar o gráfico acima, descreva o que foi observado — por
exemplo: avaliações negativas costumam ser mais longas, pois clientes
insatisfeitos tendem a detalhar o problema, enquanto avaliações
positivas costumam ser mais curtas e diretas.)_

In [ ]:
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

def tokenizer(dataframe, text_column, lower=True):
    tk = nltk.tokenize.WordPunctTokenizer()

    if lower:
        every_word = ' '.join([text.lower() for text in dataframe[text_column]])
    else:
        every_word = ' '.join([text for text in dataframe[text_column]])

    tokens = tk.tokenize(every_word)
    print(f"Token count: {len(tokens):,}")
    print(f"Unique tokens: {len(set(tokens)):,}")
    return tokens


def word_frequencies_graphic(tokenized_data, n_largest, title="Frequência de tokens"):
    frequency = nltk.FreqDist(tokenized_data)

    frequency_df = pd.DataFrame({
        "word": list(frequency.keys()),
        "frequency": list(frequency.values())
    })

    top_words = frequency_df.nlargest(columns="frequency", n=n_largest)

    plt.figure(figsize=(14, 8))
    ax = sns.barplot(data=top_words, x="word", y="frequency", palette="viridis")
    plt.title(title, fontsize=16, fontweight='bold', pad=20)
    plt.xlabel("Palavras", fontsize=12)
    plt.ylabel("Frequência", fontsize=12)
    plt.xticks(rotation=45, ha='right')

    for i, v in enumerate(top_words['frequency']):
        ax.text(i, v + 0.5, str(v), ha='center', va='bottom', fontweight='bold')

    plt.tight_layout()
    plt.show()
    return top_words

In [ ]:
tokens = tokenizer(dataset, "review_text", lower=False)
top_words = word_frequencies_graphic(tokens, 20, "Frequência de tokens (texto bruto)")

_(Descreva aqui quais palavras aparecem no topo — normalmente
serão preposições e artigos, o que reforça a necessidade de remover
stopwords antes de tirar conclusões sobre o vocabulário.)_

## 5.4 Pré-processamento

Cada tratamento aplicado abaixo é justificado individualmente,
seguindo o problema identificado, o tratamento escolhido e o motivo
da escolha.

**Problema 1 — valores ausentes.**
Textos e polaridades nulos não podem ser usados para treinar ou
avaliar um classificador de texto.
**Tratamento:** remoção das linhas com `review_text` ou `polarity`
ausentes.

**Problema 2 — textos vazios (strings em branco).**
Algumas linhas têm `review_text` como string vazia após strip, o que
não carrega nenhuma informação textual.
**Tratamento:** remoção dessas linhas.

In [ ]:
af = len(dataset)
dataset.dropna(subset=["review_text", "polarity"], inplace=True)
bf = len(dataset)
print(f"Linhas removidas por valores nulos: {af - bf:,}")

bf = len(dataset)
dataset = dataset[dataset["review_text"].str.strip().astype(bool)]
af = len(dataset)
print(f"Linhas removidas por texto vazio: {bf - af:,}")

**Problema 3 — desbalanceamento de classes.**
Como visto na seção 5.2, há muito mais avaliações positivas que
negativas. Treinar um modelo nesses dados sem correção tende a
produzir um classificador enviesado, que aprende a prever a classe
majoritária na maioria dos casos.
**Tratamento:** undersampling da classe majoritária (positiva), para
igualar o número de exemplos por classe.
**Justificativa:** o dataset é grande o suficiente para que reduzir a
classe majoritária ainda deixe exemplos suficientes para o
treinamento, e essa abordagem é mais simples e transparente do que
técnicas de oversampling sintético (como SMOTE) para dados textuais.

In [ ]:
positives = dataset.query("polarity == 1.0").shape[0]
negatives = dataset.query("polarity == 0.0").shape[0]
total = positives + negatives

print(f"% Positivos: {positives/total*100:.1f}% ({positives:,})")
print(f"% Negativos: {negatives/total*100:.1f}% ({negatives:,})")

n_neg = negatives
pos_sample = dataset.query("polarity == 1.0").sample(n=n_neg, random_state=42)
neg_sample = dataset.query("polarity == 0.0")

dataset = pd.concat([pos_sample, neg_sample], ignore_index=True)

print(f"\nApós balanceamento -> Total: {len(dataset):,}")
print(f"Negativos: {len(dataset.query('polarity == 0.0')):,}")
print(f"Positivos: {len(dataset.query('polarity == 1.0')):,}")

**Problema 4 — variação textual (maiúsculas/minúsculas, acentos,
pontuação, stopwords).**
Palavras como "Bom", "bom" e "BOM" são semanticamente idênticas, mas
seriam tratadas como tokens diferentes sem normalização. O mesmo vale
para acentuação ("não" vs "nao"). Pontuação e stopwords (artigos,
preposições) não carregam carga semântica relevante para a tarefa.
**Tratamento:** conversão para minúsculas, remoção de acentos
(unidecode), remoção de pontuação e de stopwords em português.

In [ ]:
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

puncts = [p for p in string.punctuation]
stopwords = list(set([
    unidecode.unidecode(stopword)
    for stopword in nltk.corpus.stopwords.words("portuguese")
]))
stopwords_puncts = sorted(stopwords + puncts)
print(f"Stop words: {len(stopwords)}")

word_punct_tokenizer = nltk.tokenize.WordPunctTokenizer()

def process_text(tokens):
    processed_words = []
    for word in tokens:
        processed_word = unidecode.unidecode(word.lower())
        if processed_word not in stopwords_puncts and processed_word.strip():
            processed_words.append(processed_word)
    return " ".join(processed_words)

def process_column(dataframe, text_column, output_column="treated_text"):
    treated_texts = []
    total_texts = len(dataframe)

    for i, review in enumerate(dataframe[text_column]):
        if i % 2000 == 0:
            print(f"   Processando: {i+1:,}/{total_texts:,} ({((i+1)/total_texts)*100:.1f}%)")
        tokens = word_punct_tokenizer.tokenize(review)
        treated_texts.append(process_text(tokens))

    dataframe[output_column] = treated_texts

process_column(dataset, "review_text")
print("Tratamento de texto concluído! ✅")

In [ ]:
wordcloud(dataset.query("polarity == 1.0"), "treated_text", "Nuvem de Palavras Positivas (tratado)")

In [ ]:
wordcloud(dataset.query("polarity == 0.0"), "treated_text", "Nuvem de Palavras Negativas (tratado)")

_(Compare com as nuvens da seção 5.3: agora as palavras que
aparecem são de fato relevantes para diferenciar positivo de negativo
— descreva quais palavras chamam atenção em cada classe.)_

**Problema 5 — variações morfológicas da mesma palavra (plural,
gênero, conjugação).**
Palavras como "ótimo", "ótima", "ótimos" carregam o mesmo sentido, mas
são tratadas como tokens distintos, aumentando desnecessariamente o
vocabulário e dificultando a generalização do modelo.
**Tratamento:** stemming com o `RSLPStemmer` do NLTK, específico para
a morfologia do português (diferente de stemmers genéricos como o
Porter, feito para o inglês).

In [ ]:
nltk.download('rslp')

class Stemmizer:
    def __init__(self):
        self.stemmer = nltk.RSLPStemmer()
        self.tokenizer = nltk.tokenize.WhitespaceTokenizer()

    def _stem_text(self, tokenized_data):
        return " ".join(self.stemmer.stem(word) for word in tokenized_data)

    def apply(self, dataframe, text_column, output_column="treat_stem"):
        stemmed_texts = []
        total_texts = len(dataframe)
        for i, review in enumerate(dataframe[text_column]):
            if i % 2000 == 0:
                print(f"   Processando: {i+1:,}/{total_texts:,} ({((i+1)/total_texts)*100:.1f}%)")
            tokenized_review = self.tokenizer.tokenize(review)
            stemmed_texts.append(self._stem_text(tokenized_review))
        dataframe[output_column] = stemmed_texts
        return dataframe

stemmizer = Stemmizer()
stemmizer.apply(dataset, "treated_text")
print("Stemming concluído! ✅")

In [ ]:
idx = 0
print(f"Original:   '{dataset.iloc[idx]['review_text'][:80]}...'")
print(f"Tratado:    '{dataset.iloc[idx]['treated_text'][:80]}...'")
print(f"Stemmizado: '{dataset.iloc[idx]['treat_stem'][:80]}...'")

**Problema 6 — textos que ficam vazios após o tratamento.**
Avaliações compostas só por stopwords/pontuação (ex: "muito bom!!!")
podem virar strings vazias após a remoção de stopwords.
**Tratamento:** remoção dessas linhas, pois não sobra nenhuma
informação textual utilizável.

In [ ]:
bf = len(dataset)
dataset.dropna(subset=["treat_stem"], inplace=True)
dataset = dataset[dataset["treat_stem"].str.strip().astype(bool)]
af = len(dataset)
print(f"Textos removidos por ficarem vazios após o tratamento: {bf - af:,}")

**Sobre vazamento de dados (data leakage):** diferente de uma
versão anterior deste notebook, a vetorização TF-IDF **não** é
ajustada (`fit`) neste ponto. O vetorizador só verá o vocabulário do
conjunto de treino — isso é feito explicitamente na seção 5.6, depois
da separação treino/teste da seção 5.5, para que nenhuma informação do
conjunto de teste influencie o pré-processamento.

## 5.5 Separação dos dados

Os dados foram divididos em **80% treino / 20% teste**, com
estratificação pela variável `polarity`, garantindo que a proporção
de classes (já balanceada na seção 5.4) seja preservada em ambos os
conjuntos. A proporção 80/20 foi escolhida por ser um padrão amplamente
utilizado em tarefas de classificação, oferecendo volume suficiente
para treino sem comprometer a robustez da avaliação final no conjunto
de teste, que permanece reservado e não é usado em nenhuma etapa de
ajuste do modelo ou do vetorizador.

In [ ]:
from sklearn.model_selection import train_test_split

X_train_text, X_test_text, y_train, y_test = train_test_split(
    dataset["treat_stem"],
    dataset["polarity"],
    test_size=0.2,
    stratify=dataset["polarity"],
    random_state=42,
    shuffle=True
)

print(f"Treino: {len(X_train_text):,} amostras")
print(f"Teste:  {len(X_test_text):,} amostras")

## 5.6 Modelagem

A vetorização TF-IDF é ajustada (`fit`) **apenas no conjunto de
treino**, e depois aplicada (`transform`) ao conjunto de teste — isso
evita que o vocabulário e os pesos IDF "vejam" os dados de teste,
corrigindo o vazamento de dados mencionado na seção 5.4.

Modelos utilizados:
- **Baseline**: `DummyClassifier` (sempre prevê a classe mais
  frequente), usado como referência mínima de comparação.
- **SGDClassifier**: modelo linear treinado via gradiente
  estocástico, exigido pelo edital.
- **RandomForestClassifier**: modelo de ensemble baseado em árvores,
  exigido pelo edital.
- **Regressão Logística**: modelo linear adicional, para comparação.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    lowercase=False,
    max_features=5000,
    ngram_range=(1, 2),
    token_pattern=r'\b\w+\b',
    sublinear_tf=True
)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)   # apenas transform, nunca fit

print(f"Dimensões treino: {X_train.shape}")
print(f"Dimensões teste:  {X_test.shape}")

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

models = {
    "Baseline (classe majoritária)": DummyClassifier(strategy="most_frequent", random_state=42),
    "SGDClassifier": SGDClassifier(random_state=42, max_iter=1000),
    "RandomForestClassifier": RandomForestClassifier(random_state=42, n_estimators=200, n_jobs=-1),
    "Regressão Logística": LogisticRegression(max_iter=1000, random_state=42),
}

results = {}

for name, model in models.items():
    print(f"Treinando: {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results[name] = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "y_pred": y_pred,
    }

print("Treinamento concluído! ✅")

## 5.7 Avaliação e discussão

In [ ]:
comparison_df = pd.DataFrame({
    name: {
        "Acurácia": r["accuracy"],
        "Precisão": r["precision"],
        "Revocação": r["recall"],
        "F1-score": r["f1"],
    }
    for name, r in results.items()
}).T.sort_values("F1-score", ascending=False)

comparison_df

In [ ]:
comparison_df.plot(kind="bar", figsize=(12, 6))
plt.title("Comparação de métricas entre modelos", fontsize=14, fontweight="bold")
plt.ylabel("Score")
plt.xticks(rotation=15)
plt.ylim(0, 1)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

_(Depois de rodar a tabela e o gráfico acima, escreva aqui:
qual modelo teve o melhor desempenho, por que ele foi escolhido como
modelo final, e como ele se compara ao baseline.)_

In [ ]:
best_model_name = comparison_df.index[0]
best_model = models[best_model_name]
y_pred_best = results[best_model_name]["y_pred"]

print(f"Modelo escolhido: {best_model_name}\n")
print(classification_report(y_test, y_pred_best, target_names=["Negativo", "Positivo"]))

cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Negativo", "Positivo"], yticklabels=["Negativo", "Positivo"])
plt.title(f"Matriz de Confusão — {best_model_name}", fontsize=13, fontweight="bold")
plt.xlabel("Predito")
plt.ylabel("Real")
plt.tight_layout()
plt.show()

_(Interprete a matriz de confusão: quantos falsos positivos e
falsos negativos o modelo comete, e o que isso significa na prática —
por exemplo, um falso negativo aqui seria classificar uma avaliação
negativa como positiva, o que poderia atrasar o atendimento a um
cliente insatisfeito.)_

**Discussão (preencher):**
- Qual modelo apresentou o melhor resultado e por quê.
- Quais erros foram observados (tipos de texto que o modelo mais
  erra — vale inspecionar alguns exemplos de erro manualmente).
- Quais limitações existem (ex: modelo não capta sarcasmo, negação
  complexa, gírias não vistas no treino).
- O que poderia ser melhorado (ex: embeddings pré-treinados, mais
  dados, ajuste de hiperparâmetros via GridSearch/RandomizedSearch).

In [ ]:
# Exemplo de inspeção manual de erros — ajuda a escrever a discussão acima
X_test_reset = X_test_text.reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)

errors_idx = np.where(y_test_reset.values != y_pred_best)[0][:5]

for i in errors_idx:
    print(f"Texto tratado: {X_test_reset.iloc[i]}")
    print(f"Real: {y_test_reset.iloc[i]} | Predito: {y_pred_best[i]}")
    print("-" * 60)

## Exportação do modelo final

Salvamos o vetorizador e o modelo escolhido, para reutilização futura
sem necessidade de re-treinamento.

In [ ]:
import joblib

joblib.dump(vectorizer, "tfidf_vectorizer.pkl")
joblib.dump(best_model, "modelo_final.pkl")

print(f"Modelo salvo: {best_model_name}")